In [1]:
!pip install -q "sentence-transformers>=3.0.0" datasets accelerate


In [2]:
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.evaluation import InformationRetrievalEvaluator

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

Dispositivo: cuda


In [3]:
model_id = "intfloat/multilingual-e5-large"
model = SentenceTransformer(model_id, device=device)
print(f"Modelo cargado: {model_id}")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Modelo cargado: intfloat/multilingual-e5-large


**Cargar el dataset small-spanish-legal-dataset (train / eval)**

In [4]:
dataset = load_dataset(
    "wilfredomartel/spanish-legal-dataset",
    split="train"
).shuffle(seed=42).select(range(200_000))  # 100k para el corpus

README.md:   0%|          | 0.00/3.24k [00:00<?, ?B/s]

spanish-legal-dataset.jsonl:   0%|          | 0.00/2.17G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
def process_dataset(row):
    return {
        "query": row["query"],
        "passage": row["pos"][0]
    }

new_dataset = dataset.map(process_dataset, remove_columns=["pos", "neg", "pos_score", "neg_score"])

print("Estructura del nuevo dataset:")
print(new_dataset.features)
print("\nEjemplo del nuevo dataset:")
print(new_dataset[0])

Map:   0%|          | 0/200000 [00:00<?, ? examples/s]

Estructura del nuevo dataset:
{'query': Value('string'), 'passage': Value('string')}

Ejemplo del nuevo dataset:
{'query': '¿Cuál fue la base legal y el argumento principal para la acción de protección interpuesta por el Ing. Paúl Ángel Soto Fuertes contra la Secretaría de Gestión de Riesgos en Loja?', 'passage': 'El Ing. Paúl Ángel Soto Fuertes interpuso una Acción de Protección argumentando la violación de sus derechos constitucionales a la seguridad jurídica, al debido proceso en la garantía de la motivación, y al trabajo. Fundamentó su reclamo en la notificación intempestiva y unilateral de la cesación de sus funciones como Analista de Tecnologías de la Información y Comunicación Zonal 3 (Servidor Público 7) de la Secretaría de Gestión de Riesgos en Loja. La base legal principal para su acción se encuentra en el Art. 88 de la Constitución de la República y los artículos 39 y 40 de la Ley Orgánica de Garantías Jurisdiccionales y Control Constitucional. Específicamente, el accionante

In [6]:
split_datasets = new_dataset.train_test_split(test_size=0.3, seed=42)

train_dataset = split_datasets["train"]
eval_dataset = split_datasets["test"]

print(f"Tamaño del training_dataset: {len(train_dataset)}")
print(f"Tamaño del eval_dataset_split: {len(eval_dataset)}")


Tamaño del training_dataset: 140000
Tamaño del eval_dataset_split: 60000


In [7]:
print("\nEjemplo eval[0]:")
print("Q:", eval_dataset[0]["query"])
print("A:", eval_dataset[0]["passage"][:250], "...")


Ejemplo eval[0]:
Q: ¿Cuál fue la razón principal por la que la Corte Constitucional inadmitió a trámite la consulta de constitucionalidad de norma Nro. 13-21-CN, relacionada con el procedimiento abreviado en la justicia penal juvenil?
A: La Corte Constitucional inadmitió a trámite la consulta de constitucionalidad de norma Nro. 13-21-CN, presentada por la Unidad Judicial de Adolescentes Infractores de Babahoyo, debido al incumplimiento de requisitos esenciales establecidos en su juri ...


**Construir queries, corpus, relevant_docs**

In [8]:
queries = dict(enumerate(eval_dataset["query"]))


In [9]:

# 2) Corpus: 1k pasajes de eval + 30k pasajes de train = 31k docs
corpus_passages = list(eval_dataset["passage"]) + train_dataset["passage"][:60_000]
corpus = dict(enumerate(corpus_passages))

print("Número de queries:", len(queries))        # ~60K
print("Número de docs en corpus:", len(corpus))  # ~120K

Número de queries: 60000
Número de docs en corpus: 120000


In [10]:
# 3) Qrels: la query idx tiene como doc relevante el doc idx
relevant_docs = {idx: [idx] for idx in queries}

# Ver algunos ejemplos
print("\nQuery 0:", queries[0])
print("Doc relevante para query 0 (id=0):")
print(corpus[0][:200], "...")
print("relevant_docs[0] =", relevant_docs[0])



Query 0: ¿Cuál fue la razón principal por la que la Corte Constitucional inadmitió a trámite la consulta de constitucionalidad de norma Nro. 13-21-CN, relacionada con el procedimiento abreviado en la justicia penal juvenil?
Doc relevante para query 0 (id=0):
La Corte Constitucional inadmitió a trámite la consulta de constitucionalidad de norma Nro. 13-21-CN, presentada por la Unidad Judicial de Adolescentes Infractores de Babahoyo, debido al incumplimient ...
relevant_docs[0] = [0]


**Crear el InformationRetrievalEvaluator y ejecutar evaluación**

In [11]:
dev_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="legalspanish-eval-60kq-120kd",
    show_progress_bar=True,
)

print("Evaluator creado, iniciando evaluación...")

results = dev_evaluator(model)
print("\nResultados completos:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


Evaluator creado, iniciando evaluación...


Batches:   0%|          | 0/1875 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:  33%|███▎      | 1/3 [08:19<16:38, 499.12s/it]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:  67%|██████▋   | 2/3 [16:39<08:20, 500.14s/it]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 3/3 [20:01<00:00, 400.50s/it]



Resultados completos:
legalspanish-eval-60kq-120kd_cosine_accuracy@1: 0.7594
legalspanish-eval-60kq-120kd_cosine_accuracy@3: 0.8453
legalspanish-eval-60kq-120kd_cosine_accuracy@5: 0.8699
legalspanish-eval-60kq-120kd_cosine_accuracy@10: 0.8973
legalspanish-eval-60kq-120kd_cosine_precision@1: 0.7594
legalspanish-eval-60kq-120kd_cosine_precision@3: 0.2818
legalspanish-eval-60kq-120kd_cosine_precision@5: 0.1740
legalspanish-eval-60kq-120kd_cosine_precision@10: 0.0897
legalspanish-eval-60kq-120kd_cosine_recall@1: 0.7594
legalspanish-eval-60kq-120kd_cosine_recall@3: 0.8453
legalspanish-eval-60kq-120kd_cosine_recall@5: 0.8699
legalspanish-eval-60kq-120kd_cosine_recall@10: 0.8973
legalspanish-eval-60kq-120kd_cosine_ndcg@10: 0.8296
legalspanish-eval-60kq-120kd_cosine_mrr@10: 0.8078
legalspanish-eval-60kq-120kd_cosine_map@100: 0.8106
